In [ ]:
%pip install kagglehub pandas scikit-learn matplotlib seaborn

In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Скачиваем и загружаем
path = kagglehub.dataset_download("shivamb/machine-predictive-maintenance-classification")
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)
df = pd.read_csv(full_path)

print("Данные успешно загружены!")
print(f"Размер таблицы: {df.shape}")
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Смотрим баланс классов
print("Баланс классов (0 - Работает, 1 - Поломка):")
print(df['Target'].value_counts())
print("-" * 30)

# 2. Очистка: удаляем ID и колонку-подсказку
df_clean = df.drop(columns=['UDI', 'Product ID', 'Failure Type'])

# 3. 'Type' в (One-Hot Encoding)
df_clean = pd.get_dummies(df_clean, columns=['Type'], drop_first=True)

# 4. Разделяем признаки и ответы
X = df_clean.drop(columns=['Target'])
y = df_clean['Target']

# 5. Разбиваем данные на обучающую и тестовую выборки.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Данные успешно подготовлены!")
print(f"Размер тренировочной выборки: {X_train_scaled.shape}")
print(f"Размер тестовой выборки: {X_test_scaled.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Инициализируем модели
log_reg = LogisticRegression(class_weight='balanced', random_state=42)
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)

# 2. Обучаем модели
print("Обучаем Логистическую регрессию...")
log_reg.fit(X_train_scaled, y_train)

print("Обучаем Случайный лес...")
rf_model.fit(X_train_scaled, y_train)

# 3. Делаем предсказания на тестовых данных
y_pred_log = log_reg.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)

# 4. Выводим результаты
print("\n" + "="*40)
print("РЕЗУЛЬТАТЫ: ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ")
print("="*40)
print(classification_report(y_test, y_pred_log))

print("\n" + "="*40)
print("РЕЗУЛЬТАТЫ: СЛУЧАЙНЫЙ ЛЕС (Random Forest)")
print("="*40)
print(classification_report(y_test, y_pred_rf))

# 5. Рисуем матрицу ошибок
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Работает (0)', 'Поломка (1)'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Матрица ошибок: Random Forest')
plt.show()

In [ ]:
# 1. Получаем важность признаков
importances = rf_model.feature_importances_

# 2. Создаем таблицу с названиями колонок и их важностью
feature_imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False)

# 3. Вывод результатов
print("\n" + "="*50)
print(f"{'ПРИЗНАК':<30} | {'ВАЖНОСТЬ (ВЕС)':<15}")
print("-" * 50)

for index, row in feature_imp_df.iterrows():
    # Печатаем название признака (выравнивание по левому краю) и его вес (4 знака после запятой)
    print(f"{row['Feature']:<30} | {row['Importance']:.4f}")

print("="*50)
top_feature = feature_imp_df.iloc[0]['Feature']
print(f"\nВывод: Самым влиятельным фактором поломки является '{top_feature}'.")

# 4. Строим график
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df, hue='Feature', palette='Reds_r', legend=False)
plt.title('Важность признаков (Графическое представление)')
plt.xlabel('Вес признака')
plt.ylabel('Признак')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Улучшаем модель

# Копируем исходный датасет
df_smart = df.copy()

# Создаем новый физический признак "Мощность"
df_smart['Power'] = df_smart['Torque [Nm]'] * df_smart['Rotational speed [rpm]']

# Выкидываем мусор
df_smart = df_smart.drop(columns=['UDI', 'Product ID', 'Failure Type', 'Type'])

# Разделяем
X_smart = df_smart.drop(columns=['Target'])
y_smart = df_smart['Target']
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_smart, y_smart, test_size=0.2, random_state=42, stratify=y_smart)

# Масштабируем
scaler_s = StandardScaler()
X_train_scaled_s = scaler_s.fit_transform(X_train_s)
X_test_scaled_s = scaler_s.transform(X_test_s)


# Обучаем
rf_smart = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf_smart.fit(X_train_scaled_s, y_train_s)

y_pred_smart = rf_smart.predict(X_test_scaled_s)

print("="*50)
print("РЕЗУЛЬТАТЫ: УЛУЧШЕННЫЙ RANDOM FOREST")
print("="*50)
print(classification_report(y_test_s, y_pred_smart))

# Оценка важности новых признаков
importances_smart = rf_smart.feature_importances_
feature_imp_df_smart = pd.DataFrame({
    'Feature': X_smart.columns,
    'Importance': importances_smart
}).sort_values(by='Importance', ascending=False)

print("\nНОВАЯ ВАЖНОСТЬ ПРИЗНАКОВ:")
print(feature_imp_df_smart.to_string(index=False))

# График
plt.figure(figsize=(8, 4))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df_smart, hue='Feature', palette='Reds_r', legend=False)
plt.title('Важность признаков (После Feature Engineering)')
plt.xlabel('Вес признака')
plt.show()